# Database export exploration

Use this notebook after exporting a focused CSV sample from phpMyAdmin. Place exported CSV files in `data/raw/`. Do not commit real exports if they may contain sensitive information.

## Workflow

1. Run a focused SQL query in phpMyAdmin.
2. Export the result as CSV.
3. Place the CSV in `data/raw/`.
4. Load it with pandas.
5. Calculate threshold metrics for MegaDetector human confidence scores.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT / "src"))

from mammalweb_analysis.io import list_raw_csvs, load_raw_csv
from mammalweb_analysis.thresholds import threshold_metrics, add_human_filter_flag

sns.set_theme(style="whitegrid")

## Find exported CSV files

In [ ]:
csv_files = list_raw_csvs(PROJECT_ROOT / "data" / "raw")
csv_files

## Load one export

Change `CSV_NAME` to match one of the files listed above.

In [ ]:
CSV_NAME = "replace_with_export_name.csv"

df = load_raw_csv(CSV_NAME, data_dir=PROJECT_ROOT / "data" / "raw")
df.head()

In [ ]:
df.shape, df.columns.tolist()

## Threshold metrics

Update these column names after inspecting the CSV columns. `SCORE_COLUMN` should contain MegaDetector human confidence scores. `ACTUAL_HUMAN_COLUMN` should contain the ground-truth human label from review/manual classification.

In [ ]:
SCORE_COLUMN = "human_confidence"
ACTUAL_HUMAN_COLUMN = "actual_human"

metrics = threshold_metrics(df, SCORE_COLUMN, ACTUAL_HUMAN_COLUMN)
metrics.head()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sns.lineplot(data=metrics, x="threshold", y="precision", marker="o", label="Precision", ax=ax)
sns.lineplot(data=metrics, x="threshold", y="recall", marker="o", label="Recall", ax=ax)
ax.set_title("MegaDetector human filtering threshold metrics")
ax.set_xlabel("Human confidence threshold")
ax.set_ylabel("Score")
ax.set_ylim(0, 1)
plt.show()

## Apply one threshold

Use this to inspect which images would be filtered as human at a chosen threshold.

In [ ]:
SELECTED_THRESHOLD = 0.5
flagged = add_human_filter_flag(df, SCORE_COLUMN, SELECTED_THRESHOLD)
flagged.head()